# 00 - Build the analysis datasets

Everything downstream reads from three processed tables. This notebook creates
them from the raw field records and writes them to `data/processed/`.

| Output | Shape | Content |
|---|---|---|
| `ppfd_long.csv` | one row per reading | PPFD, 10-min steps, 11 campaigns x 5 sensor positions |
| `rfr_long.csv` | one row per reading | R:FR ratio, same design |
| `unified_growth_dataset.csv` | 2 250 rows | 90 principal axes x 25 months, with climate and light features |
| `physiology_clean.csv` | one row per leaf measurement | gas exchange plus derived efficiency traits |

Run this notebook once; notebooks 01-06 are then independent of each other.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from yerbamate import config as C
from yerbamate import io_growth, io_light, io_physio

pd.set_option("display.width", 160, "display.max_columns", 40)
print("repository root:", C.ROOT)

## 1. Light campaigns -> long format

The raw files are wide matrices: one column per (measurement day x sensor
position), one row per 10-minute time step. `build_long` melts each cell into a
row, keeping the measurement-day identity so the daily light integral can be
computed later.

Light intensity (PPFD, 400-700 nm) came from an LI-190R quantum sensor; light
quality (R:FR) from an SKR 110 sensor reading 655-665 nm and 725-735 nm. Both
logged 10-second readings integrated over 10 minutes on an LI-1400.

In [ ]:
ppfd = io_light.build_long(C.RAW_PPFD, "PPFD")
ppfd.to_csv(C.PPFD_LONG, index=False)
print(f"\nPPFD: {len(ppfd):,} readings")
print(ppfd.groupby("Environment").size().to_string())

In [ ]:
rfr = io_light.build_long(C.RAW_RFR, "R_FR")
rfr.to_csv(C.RFR_LONG, index=False)
print(f"\nR:FR: {len(rfr):,} readings")
print(rfr.groupby("Environment").size().to_string())

## 2. Unified growth dataset

Thirty plants (15 per cultivation system) each carried three tagged buds, giving
90 principal axes followed monthly for 25 months (Jun 2003 - Jun 2005). Five
traits were measured per axis and month: shoot elongation, metamer emission,
leaf number increase, leaf area increase and leaf shed.

To each axis-month record we attach the 13 environmental features used in the analysis:
GDD, Tmax, Tmin, DTR, night length, precipitation, and the seven light features
(midday / morning / afternoon PPFD and R:FR, plus DLI). Light was measured at
only 11 of the 25 months, so it is linearly interpolated across the series.

In [ ]:
growth = io_growth.build_unified(ppfd, rfr)
growth.to_csv(C.UNIFIED, index=False)

print(f"unified dataset: {growth.shape[0]:,} rows x {growth.shape[1]} columns")
print(f"  axes: {growth.axis_id.nunique()}   plants: {growth.plant_id.nunique()}   months: {growth.time_idx.nunique()}")
print("\naxis counts by system and sex (should be MO 30F/15M, AFS 12F/33M):")
print(growth.groupby(["environment", "sex"]).axis_id.nunique().to_string())

In [ ]:
# Growth/rest phase structure from Guedon et al. (2018).
print("months per phase:")
print(growth.groupby("time_idx").phase_label.first().value_counts().to_string())

print("\nmean light per system (crown-top 2 m sensor):")
print(growth.groupby("environment")[io_growth.LIGHT_COLS].mean().round(2).to_string())

## 3. Leaf gas exchange

Measured in situ with an LI-6200 at the same eleven bimonthly campaigns, on
tagged leaves of different ages, during the 10:00-14:30 window when diurnal
assimilation varies least. Recorded: A, gs, E, PPFD, leaf and air temperature.
Derived here: WUE = A/E, iWUE = A/gs, LUE = A/PPFD and deltaT = Tleaf - Tair.

In [ ]:
physio = io_physio.load_physiology()
physio.to_csv(C.PHYSIO_CLEAN, index=False)

print(f"physiology: {physio.shape[0]:,} leaf measurements, {physio.Folha.nunique()} unique leaves")
print("\nrecords by system and sex:")
print(physio.groupby(["environment", "Sexo"]).size().to_string())
print("\nderived trait coverage:")
print(physio[C.PHYSIO_VARS].notna().sum().to_string())

## Checks

These assertions validate the dataset shapes generated for the manuscript analysis. If a raw
file is replaced they will fail loudly rather than silently changing the source results.

In [ ]:
assert len(growth) == 2250, f"expected 2250 axis-month records, got {len(growth)}"
assert growth.axis_id.nunique() == 90
assert set(growth.environment) == {"MO", "FUS"}
assert C.PPFD_LONG.exists() and C.RFR_LONG.exists()
assert C.UNIFIED.exists() and C.PHYSIO_CLEAN.exists()
print("all datasets built and verified ->", C.DATA_PROC)